# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guided exploration of a Croissant-structured dataset using the `mlcroissant` library, following reproducible FAIR principles.

### Dataset Source
The dataset is defined via a Croissant schema and available at the URL below.


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and inspect basic information using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Metadata is accessible as an object, not as a dictionary
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")

## 2. Data Overview
Review available record sets and fields by ID. Below, we enumerate record sets, their `@id`s, and which fields are available. Use these `@id` values for data extraction.

In [ ]:
# List all record sets and their @id fields
record_sets = list(dataset.record_sets)
print("Record sets available in the dataset:")
for rs in record_sets:
    print(f"- name: {rs.name}")
    print(f"  @id: {rs['@id']}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field['@id']})")
    print("")

## 3. Data Extraction
Load data from each record set into a pandas DataFrame. All record sets and fields are referenced via their Croissant `@id`.

This example loads the first record set for exploration. You may update the `record_set_id` and field IDs below according to your needs.

In [ ]:
# Gather all record set @id values
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Print out all DataFrames and their columns
for rs_id in record_set_ids:
    print(f"Record Set @id: {rs_id}")
    df = dataframes[rs_id]
    print(f"Columns: {list(df.columns)}\nShape: {df.shape}")
    display(df.head())
    print("-")

## 4. Exploratory Data Analysis (EDA)
This example explores a numeric field by filtering, normalizing, and grouping. Select the record set and field `@id` values (from previous cells) to adapt this example to the dataset.

**Note:** The field and group IDs below must correspond to actual `@id`s from the Croissant metadata.

In [ ]:
# Example: Use the first record set and attempt to process a numeric column.

# Select specific record set and fields (update these as per available IDs shown above)
record_set_id = record_set_ids[0]  # update index if needed
df = dataframes[record_set_id]

# Choose a numeric field for exploration (select a real field @id)
numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
print(f"Numeric fields in {record_set_id}:", numeric_fields)
if numeric_fields:
    numeric_field = numeric_fields[0]
    threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())
    
    # Normalize field
    normalized = (filtered_df[numeric_field] - filtered_df[numeric_field].mean())/filtered_df[numeric_field].std()
    filtered_df[numeric_field + "_normalized"] = normalized
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, numeric_field + "_normalized"]].head())
    
    # Try grouping by a categorical column if available
    group_fields = [col for col in df.columns if pd.api.types.is_object_dtype(df[col]) and col != numeric_field]
    if group_fields:
        group_field = group_fields[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        print(f"Grouped data by {group_field} (mean {numeric_field}):")
        print(grouped_df.head())
else:
    print(f"No numeric field detected in record set {record_set_id} for EDA.")

## 5. Visualization
Visualize the distribution of a numeric column and/or relationships. Edit the field and record set choices to suit the dataset's structure.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# If a DataFrame and a normalized numeric field exist, plot
if numeric_fields:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.xlabel(numeric_field)
    plt.title(f"Distribution of {numeric_field}")
    plt.show()
    
    # If grouped analysis available
    if 'group_field' in locals():
        plt.figure(figsize=(10,6))
        sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
        plt.xticks(rotation=45)
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()

## 6. Conclusion
We successfully loaded a FAIR dataset using `mlcroissant`, examined the available record sets and fields by `@id`, and performed basic exploratory and visual analysis. Adjust fields and analyses as needed to suit your data science questions on this dataset!